IMPORTS

In [29]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from collections import defaultdict

LOAD DATA

In [30]:
splits = {
    "train": "data/train-00000-of-00001.parquet",
    "validation": "data/validation-00000-of-00001.parquet",
    "test": "data/test-00000-of-00001.parquet"
}

train_df = pd.read_parquet("hf://datasets/surrey-nlp/BESSTIE-CW-26/" + splits["train"])
val_df = pd.read_parquet("hf://datasets/surrey-nlp/BESSTIE-CW-26/" + splits["validation"])
test_df = pd.read_parquet("hf://datasets/surrey-nlp/BESSTIE-CW-26/" + splits["test"])

label_col = "Sentiment"

train_df[label_col] = train_df[label_col].astype(int)
val_df[label_col] = val_df[label_col].astype(int)
test_df[label_col] = test_df[label_col].astype(int)

TOKENIZER

In [31]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

def convert_to_dataset(df):
    dataset = Dataset.from_dict({
        "text": df["text"].tolist(),
        "label": df[label_col].tolist()
    })

    dataset = dataset.map(tokenize, batched=True)


    dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

    return dataset

SETTINGS

In [32]:
varieties = ["en-AU", "en-UK", "en-IN"]
num_runs = 2

macro_f1_scores = defaultdict(list)

METRICS

In [4]:
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average='macro')
    }

MAIN FUNCTION

In [34]:
for run in range(num_runs):
    print(f"\n===== RUN {run+1} =====")

    for train_var in varieties:
        print(f"\nTraining on {train_var}")

        train_subset = train_df[train_df["variety"] == train_var]
        val_subset = val_df[val_df["variety"] == train_var]

        train_dataset = convert_to_dataset(train_subset)
        val_dataset = convert_to_dataset(val_subset)


        model = AutoModelForSequenceClassification.from_pretrained(
            "bert-base-uncased",
            num_labels=2,
            problem_type="single_label_classification"
        )


        training_args = TrainingArguments(
            output_dir=f"./results_{train_var}_{run}",
            num_train_epochs=2,
            per_device_train_batch_size=8,
            per_device_eval_batch_size=8,
            logging_dir="./logs",
            seed=42 + run
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset
        )

        trainer.train()


        for test_var in varieties:
            test_subset = test_df[test_df["variety"] == test_var]
            test_dataset = convert_to_dataset(test_subset)

            predictions = trainer.predict(test_dataset)

            preds = np.argmax(predictions.predictions, axis=1)
            labels = predictions.label_ids.astype(int)

            acc = accuracy_score(labels, preds)
            macro_f1 = f1_score(labels, preds, average='macro')

            print(f"\nTrain {train_var} → Test {test_var}")
            print("Accuracy:", acc)
            print("Macro F1:", macro_f1)

            print("\nClassification Report:")
            print(classification_report(labels, preds))

            cm = confusion_matrix(labels, preds)
            print("Confusion Matrix:\n", cm)


            macro_f1_scores[(train_var, test_var)].append(macro_f1)


===== RUN 1 =====

Training on en-AU


Map:   0%|          | 0/1145 [00:00<?, ? examples/s]

Map:   0%|          | 0/95 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will 

...


Train en-IN → Test en-AU
Accuracy: 0.863568215892054
Macro F1: 0.8619597229960996

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.93      0.88       347
           1       0.92      0.79      0.85       320

    accuracy                           0.86       667
   macro avg       0.87      0.86      0.86       667
weighted avg       0.87      0.86      0.86       667

Confusion Matrix:
 [[324  23]
 [ 68 252]]


Map:   0%|          | 0/700 [00:00<?, ? examples/s]


Train en-IN → Test en-UK
Accuracy: 0.9328571428571428
Macro F1: 0.9327713920817369

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.92      0.93       340
           1       0.93      0.94      0.94       360

    accuracy                           0.93       700
   macro avg       0.93      0.93      0.93       700
weighted avg       0.93      0.93      0.93       700

Confusion Matrix:
 [[314  26]
 [ 21 339]]


Map:   0%|          | 0/816 [00:00<?, ? examples/s]


Train en-IN → Test en-IN
Accuracy: 0.8394607843137255
Macro F1: 0.8393910755791678

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.82      0.84       430
           1       0.81      0.87      0.84       386

    accuracy                           0.84       816
   macro avg       0.84      0.84      0.84       816
weighted avg       0.84      0.84      0.84       816

Confusion Matrix:
 [[351  79]
 [ 52 334]]


STORAGE CHECK

In [35]:
print("\nStored Keys:")
print(list(macro_f1_scores.keys()))


Stored Keys:
[('en-AU', 'en-AU'), ('en-AU', 'en-UK'), ('en-AU', 'en-IN'), ('en-UK', 'en-AU'), ('en-UK', 'en-UK'), ('en-UK', 'en-IN'), ('en-IN', 'en-AU'), ('en-IN', 'en-UK'), ('en-IN', 'en-IN')]


RESULTS

In [36]:
mean_matrix = pd.DataFrame(index=varieties, columns=varieties)
std_matrix = pd.DataFrame(index=varieties, columns=varieties)

for train_var in varieties:
    for test_var in varieties:
        scores = macro_f1_scores[(train_var, test_var)]

        if len(scores) > 0:
            mean_matrix.loc[train_var, test_var] = round(np.mean(scores), 3)
            std_matrix.loc[train_var, test_var] = round(np.std(scores), 3)

print("\n=== MEAN MATRIX ===")
print(mean_matrix)

print("\n=== STD MATRIX ===")
print(std_matrix)


final_matrix = pd.DataFrame(index=varieties, columns=varieties)

for train_var in varieties:
    for test_var in varieties:
        mean = mean_matrix.loc[train_var, test_var]
        std = std_matrix.loc[train_var, test_var]

        if pd.notna(mean):
            final_matrix.loc[train_var, test_var] = f"{mean} ± {std}"

print("\n=== FINAL CROSS-VARIETY MATRIX (Mean ± Std) ===")
print(final_matrix)


=== MEAN MATRIX ===
       en-AU  en-UK  en-IN
en-AU  0.891  0.928  0.796
en-UK  0.866  0.936  0.815
en-IN  0.857  0.931  0.839

=== STD MATRIX ===
       en-AU  en-UK  en-IN
en-AU  0.004  0.002  0.001
en-UK  0.013    0.0  0.015
en-IN  0.005  0.001    0.0

=== FINAL CROSS-VARIETY MATRIX (Mean ± Std) ===
               en-AU          en-UK          en-IN
en-AU  0.891 ± 0.004  0.928 ± 0.002  0.796 ± 0.001
en-UK  0.866 ± 0.013    0.936 ± 0.0  0.815 ± 0.015
en-IN  0.857 ± 0.005  0.931 ± 0.001    0.839 ± 0.0
